# Geracao de prompts para LLM por interseccao

Este notebook gera prompts completos e prompts em lotes para cada interseccao presente em `base_sem_duplicados.csv`.

Saida:
- `dados/prompts/<interseccao>/prompt_<interseccao>_completo.txt`
- `dados/prompts/<interseccao>/lotes/prompt_<interseccao>_lote_XXX.txt`

In [21]:
from pathlib import Path
import json
import math
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data_path = repo_root / 'dados' / 'identificados' / 'base_sem_duplicados.csv'
prompts_root = repo_root / 'dados' / 'prompts'

df = pd.read_csv(data_path)
df['denominacao_intersecao'] = df['denominacao_intersecao'].astype(str).str.upper()

intersecoes_disponiveis = sorted(df['denominacao_intersecao'].dropna().unique().tolist())

print(f'Arquivo lido: {data_path}')
display(pd.DataFrame({'interseccao_disponivel': intersecoes_disponiveis}))

Arquivo lido: c:\Users\ultra\OneDrive\Documentos\github\revisor\dados\identificados\base_sem_duplicados.csv


,interseccao_disponivel
0,AG
1,CA
2,CAS
3,CS


In [22]:
df = df[df['publication_year'] > 2014]
df['publication_year'].unique()

array([2025, 2015, 2016, 2021, 2019, 2024, 2022, 2026, 2018, 2023, 2017,
       2020])

In [23]:
# Ajuste o tamanho dos lotes se necessario
tamanho_lote = 50

PROMPTS_INTERSECCAO = {
    'CA': {
        'titulo': "Consumers' Interpretation of AI (AI Literacy)",
        'contexto': [
            'How consumers interpret AI systems (chatbots, virtual assistants, AI agents)',
            'AI literacy and understanding of AI capabilities',
            'Trust, perception, attitudes, and cognitive evaluation of AI',
            'Attribution of responsibility and interpretation of automated decisions'
        ],
        'exclusoes': [
            'Conference proceedings, book chapters, editorials, commentaries, working papers',
            'Purely technical/engineering AI studies (e.g., algorithm development)',
            'Studies without business/management implications',
            'Articles focused only on B2C without relevance to B2B or interorganizational context',
            'Studies about AI that do NOT involve consumer perception or interpretation',
            'Articles where AI is mentioned only peripherally',
            'Studies not related to service contexts',
            'Duplicate records'
        ]
    },
    'AG': {
        'titulo': 'AI Agents, Governance and Decision Alignment',
        'contexto': [
            'Use of AI agents in service recovery processes',
            'Governance, control, monitoring, and accountability of AI decisions',
            'Alignment between AI decisions and organizational objectives',
            'Delegation of decision-making to AI agents',
            'Agency-related issues (autonomy, misalignment, control)'
        ],
        'exclusoes': [
            'Conference proceedings, book chapters, editorials, commentaries, working papers',
            'Purely technical AI or algorithm-focused studies',
            'Studies without organizational or managerial implications',
            'Articles about AI in general service contexts WITHOUT service recovery or failure',
            'B2C-only studies without relevance to interorganizational/B2B context',
            'Articles that do not address decision-making, governance, or control',
            'Peripheral mention of AI or service recovery',
            'Duplicate records'
        ]
    },
    'CS': {
        'titulo': 'Consumer Experience and Service Recovery',
        'contexto': [
            'Consumer reactions to service failure and recovery',
            'Perceived justice, satisfaction, trust, and expectations',
            'Behavioral and emotional responses after service recovery',
            'Customer engagement and relationship outcomes'
        ],
        'exclusoes': [
            'Conference proceedings, book chapters, editorials, commentaries, working papers',
            'Purely technical or engineering studies',
            'Articles that do not involve consumers or customer perception',
            'Studies about AI WITHOUT connection to service recovery',
            'Articles focused exclusively on B2C without conceptual relevance to B2B',
            'Studies not involving service failure or recovery',
            'Peripheral treatment of service recovery',
            'Duplicate records'
        ]
    },
    'CAS': {
        'titulo': 'AI-mediated Service Recovery',
        'contexto': [
            'AI agents in service recovery contexts',
            'Consumer interpretation of AI decisions',
            'Organizational implications (governance, alignment, agency)',
            'Trust, perceived justice, and engagement in AI-mediated service recovery',
            'Delegation of decisions to AI agents',
            'B2B relationships and interorganizational effects',
            'Interaction between AI capabilities, governance, and consumer perception'
        ],
        'exclusoes': [
            'Conference proceedings, book chapters, editorials, commentaries, working papers',
            'Purely technical AI or algorithm-focused studies',
            'Articles that do not integrate at least two domains',
            'Studies without service recovery context',
            'Studies without consumer interpretation or perception',
            'Articles without organizational or governance implications',
            'B2C-only studies without relevance to B2B/interorganizational relationships',
            'Peripheral treatment of the core constructs',
            'Duplicate records'
        ]
    }
}

faltantes = [i for i in intersecoes_disponiveis if i not in PROMPTS_INTERSECCAO]
if faltantes:
    raise ValueError(f'Faltam configuracoes de prompt para: {faltantes}')

print(f'Tamanho de lote: {tamanho_lote}')

Tamanho de lote: 50


In [24]:
def limpar_texto(valor):
    if pd.isna(valor):
        return None
    texto = str(valor).replace('\r', ' ').replace('\n', ' ')
    return ' '.join(texto.split())

def montar_registros(df_base, prefixo):
    registros = []
    for idx, row in df_base.reset_index(drop=True).iterrows():
        registros.append({
            'article_id': f'{prefixo}_{idx + 1:04d}',
            'title': limpar_texto(row.get('title')),
            'journal': limpar_texto(row.get('journal')),
            'journal_impact_factor': None if pd.isna(row.get('journal_impact_factor')) else float(row.get('journal_impact_factor')),
            'doi': limpar_texto(row.get('doi')),
            'abstract': limpar_texto(row.get('abstract'))
        })
    return registros

def bullets(linhas):
    return '\n'.join(f'- {linha}' for linha in linhas)

def montar_prompt(registros, interseccao, config):
    titulo = config['titulo']
    contexto = bullets(config['contexto'])
    exclusoes = bullets(config['exclusoes'])

    return f"""You are an expert researcher supporting a systematic literature review using PRISMA and the Lotus Protocol.

This prompt focuses ONLY on the {interseccao} intersection:
{titulo}.

This intersection includes studies that analyze:
{contexto}

You will receive a LIST of academic articles. Each article contains:
- article_id
- title
- journal
- journal_impact_factor
- doi
- abstract

Your task is to analyze EACH article independently.

Instructions:
1. Determine whether the article is relevant to the {interseccao} intersection.
2. If relevant:
   - set \"relevance\" to \"relevant\"
   - set \"intersection\" to \"{interseccao}\"
   - provide summary in English and Portuguese
   - extract relevant excerpts in English and Portuguese
3. If not relevant:
   - set \"relevance\" to \"not_relevant\"
   - set \"intersection\" to null
   - justify clearly why the article should be excluded

EXCLUSION CRITERIA:
{exclusoes}

DECISION RULES:
- Use only the information available in the title and abstract
- Be conservative and classify as relevant only when the article clearly matches the intersection scope
- Preserve the input metadata exactly as received
- Return one JSON object for each input article
- Keep the same order as the input list
- Return ALL articles from the input list in the output
- Return ONLY a valid JSON array
- Do not include markdown fences
- Do not include commentary before or after the JSON array

OUTPUT FORMAT FOR EACH ARTICLE:
{{
  \"article_id\": \"...\",
  \"title\": \"...\",
  \"journal\": \"...\",
  \"journal_impact_factor\": \"...\",
  \"doi\": \"...\",
  \"relevance\": \"relevant\" or \"not_relevant\",
  \"intersection\": \"{interseccao}\" or null,
  \"summary_en\": \"...\",
  \"summary_pt\": \"...\",
  \"relevant_excerpts_en\": [\"...\", \"...\"],
  \"relevant_excerpts_pt\": [\"...\", \"...\"],
  \"justification\": \"...\",
  \"confidence\": 0.0
}}

ARTICLE LIST TO ANALYZE:
{json.dumps(registros, ensure_ascii=False, indent=2)}
"""


In [25]:
resumo_geracao = []

for interseccao in intersecoes_disponiveis:
    config = PROMPTS_INTERSECCAO[interseccao]
    base_interseccao = df[df['denominacao_intersecao'].eq(interseccao)].copy()
    base_interseccao = base_interseccao[base_interseccao['title'].notna() & base_interseccao['abstract'].notna()].reset_index(drop=True)

    registros_total = montar_registros(base_interseccao, prefixo=interseccao)
    prompt_completo = montar_prompt(registros_total, interseccao, config)

    interseccao_dir = prompts_root / interseccao.lower()
    lotes_dir = interseccao_dir / 'lotes'
    interseccao_dir.mkdir(parents=True, exist_ok=True)
    lotes_dir.mkdir(parents=True, exist_ok=True)

    prompt_completo_path = interseccao_dir / f'prompt_{interseccao.lower()}_completo.txt'
    prompt_completo_path.write_text(prompt_completo, encoding='utf-8')

    total_lotes = math.ceil(len(registros_total) / tamanho_lote) if registros_total else 0
    for inicio in range(0, len(registros_total), tamanho_lote):
        fim = min(inicio + tamanho_lote, len(registros_total))
        lote = registros_total[inicio:fim]
        numero_lote = (inicio // tamanho_lote) + 1
        prompt_lote = montar_prompt(lote, interseccao, config)
        lote_path = lotes_dir / f'prompt_{interseccao.lower()}_lote_{numero_lote:03d}.txt'
        lote_path.write_text(prompt_lote, encoding='utf-8')

    resumo_geracao.append({
        'interseccao': interseccao,
        'artigos': len(registros_total),
        'prompt_completo': str(prompt_completo_path.relative_to(repo_root)),
        'tamanho_prompt_completo': len(prompt_completo),
        'total_lotes': total_lotes,
        'pasta_lotes': str(lotes_dir.relative_to(repo_root))
    })

df_resumo_geracao = pd.DataFrame(resumo_geracao)
display(df_resumo_geracao)

,interseccao,artigos,prompt_completo,tamanho_prompt_completo,total_lotes,pasta_lotes
0,AG,10,dados\prompts\ag\prompt_ag_completo.txt,18059,1,dados\prompts\ag\lotes
1,CA,2426,dados\prompts\ca\prompt_ca_completo.txt,4209398,49,dados\prompts\ca\lotes
2,CAS,89,dados\prompts\cas\prompt_cas_completo.txt,151857,2,dados\prompts\cas\lotes
3,CS,2789,dados\prompts\cs\prompt_cs_completo.txt,4850965,56,dados\prompts\cs\lotes
